In [10]:
import os
import sys

repo_root_path = os.path.abspath(os.path.join("..", "..", ".."))

if repo_root_path not in sys.path:
    sys.path.append(repo_root_path)


In [11]:
from analyse.advertising.e01_rupture_detection.rupture_detector import (
    RuptureDetector,
    print_summary,
)
from dateutil import tz
from datetime import datetime
from analyse.advertising.tools.testimony_table.extract import get_testimony_data
from analyse.advertising.tools.segment_player.player_generator import (
    format_annotation,
    generate_player,
)

tz_paris = tz.gettz("Europe/Paris")
start_time = datetime(2025, 3, 10, 12, 0, tzinfo=tz_paris)
end_time = datetime(2025, 3, 10, 12, 30, tzinfo=tz_paris)

CHANNEL = "fr3-idf"
TESTIMONY_CHANNEL = "FRANCE 3"

annotations = get_testimony_data(
    channel=TESTIMONY_CHANNEL,
    from_date=start_time,
    to_date=end_time,
    source_file="export.csv",
)

detector = RuptureDetector(
    sensitivity=0.1,
    context_sec=1.0,  # durée analysée de chaque côté d'un point pour évaluer la rupture.
    max_ruptures=0,  # 0 pour pas de limite
    sr=22050,  # Fréquence d'échantillonnage de travail
    hop_length=512,  # Pas entre frames (~23ms à 22050Hz)
    n_mfcc=10,  # Nombre de coefficients MFCC
    novelty_smooth_sec=0.5,  # Lissage temporel de la courbe de nouveauté.
    min_segment_sec=1.0,  # Durée minimale d'un segment pour être considéré comme tel.
    silence_percentile=8.0,  # Percentile pour définir le seuil de silence
    cosine_weight=1.0,
)

len(annotations)

1

In [12]:
from analyse.advertising.e00_download.mediatree import CachedMediatreeAPI
import httpx

mediatree_api = CachedMediatreeAPI()
input_file = await mediatree_api.download_export(CHANNEL, start_time, end_time)
async with httpx.AsyncClient(timeout=httpx.Timeout(60.0, connect=60.0)) as client:
    witness_file = await mediatree_api.api.get_single_export_url(client, CHANNEL, start_time, end_time, media_format="mp4")

# input_file = "/root/Workspace/quotaclimat/analyse/advertising/exports/tf1_2025-03-04_12-00-00_2025-03-04_13-00-00.mp3"
# witness_file = (
#     "https://vod.mediatree.fr/assets/tf1_2025-03-04T12-00-00Z_2025-03-04T13-00-00Z.mp4"
# )

In [13]:
segments, peaks, novelty, features, y = detector.run(input_file)

print_summary(segments)

generate_player(
    media_input=witness_file,
    segments=[s.to_dict() for s in segments],
    annotations=format_annotation(annotations, from_date=start_time),
    output_path="output.html",
    params=detector.get_params(),
    novelty_peaks=detector.get_novelty_peaks(novelty),
    start_epoch=int(start_time.timestamp()),
)

[1/5] Chargement : ../exports/fr3-idf_2025-03-10_11-00-00Z_2025-03-10_11-30-00Z.mp3
      Durée : 1800.0s  (30.0 min)
      → 4.19s
[2/5] Extraction des features...
      → 3.90s
[3/5] Calcul de la courbe de nouveauté...
      Fenêtre de contexte : 43 frames = 1.00s de chaque côté
      Seuil silence       : énergie < 0.00537 (percentile 8.0) → 11959 frames silencieuses (15.4%)
      Lissage             : 21 frames = 0.49s
      → 1.62s
[4/5] Détection des frontières...
      Seuil sensitivity=0.10 → hauteur min 0.0061 (percentile 90)
      Après seuil + distance min (1.0s) : 259 ruptures candidates
      Résultat final : 259 ruptures → 260 segments
      → 0.00s
[5/5] Segmentation terminée : 260 segments  → 0.01s
      Durée totale : 9.72s

═══════════════════════════════════════════════════════
  RÉSUMÉ DE SEGMENTATION
═══════════════════════════════════════════════════════
  Total segments : 260
  Label                     Nb       %   Durée moy
  ───────────────────────────────────